# Các chiến thuật mã hóa dữ liệu (ENCODING) trên bộ dữ liệu Walmart Recruiting Store Sales Forecasting

Mã hóa dữ liệu (encoding) là phương pháp xử lý các biến phân loại (categorical variables) giúp mô hình học máy hiểu và khai thác tốt hơn.

Trước tiên, ta import các thư viện cần thiết:

In [27]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

Tiếp theo như thường lệ, ta phải chuẩn bị bộ dữ liệu:

In [28]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
features = pd.read_csv('features.csv')
stores = pd.read_csv('stores.csv')

Xem qua một vài dòng của các tập dữ liệu:

In [29]:
train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [30]:
test.head()

,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [31]:
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [32]:
stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


Mã hóa dữ liệu cần được thực hiện trên cả tập train lẫn tập test, hay nói cách khác là trên tất cả các tập dữ liệu mà mô hình máy học phải làm việc cùng. Vì vậy, ta sẽ phải gom chung các tập dữ liệu (bao gồm cả tập test) để tiến hành các chiến thuật mã hóa dữ liệu. Việc phân chia lại các tập train, test sẽ được thực hiện trong quá trình xây dựng model, sau khi đã thực hiện xong tiền xử lý dữ liệu.

Gom chung các tập dữ liệu thành data frame df:

In [33]:
# Kết hợp hai tập train và test:
# Thêm cột set để phân biệt:
train['Set'] = 'train'
test['Set'] = 'test'
# Thêm cột Weekly_Sales tạm thời cho test:
test['Weekly_Sales'] = np.nan
# Gộp hai tập:
combined = pd.concat([train, test], ignore_index=True)
# Kết hợp combined với features dựa trên Store, Date và IsHoliday:
df = pd.merge(combined, features, on=['Store', 'Date', 'IsHoliday'], how='left')
# Kết hợp bảng stores với df dựa trên cột stores:
df = pd.merge(df, stores, on='Store', how='left')

In [34]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Set,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,train,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,train,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,train,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,train,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,train,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [35]:
df['Set'].value_counts()

Set
train    421570
test     115064
Name: count, dtype: int64

Như vậy, ta đã có một dataframe hoàn chỉnh với cả tập train và test, sau khi thực hiện tiền xử lý dữ liệu, ta có thể phân loại chính xác tập train và test dựa trên cột Set đã được thêm vào.

**Bây giờ ta sẽ tiến hành mã hóa dữ liệu (ENCODING):**

# 1. Label Encoding (Mã hóa nhãn)

Mã hóa nhãn là chiến thuật mã hóa chuyển dữ liệu dạng phân loại thành số nguyên, phù hợp với các đặc trưng phân loại chỉ nhận ít giá trị. Trong bộ dữ liệu này, ta có thể mã hóa cột `Type` từ A, B, C thành 0, 1, 2; hay cột `IsHoliday` từ True, False thành 1, 0.

In [36]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Type_enc'] = le.fit_transform(df['Type'])
df['IsHoliday_enc'] = le.fit_transform(df['IsHoliday'])

In [37]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Set,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Type_enc,IsHoliday_enc
0,1,1,2010-02-05,24924.50,False,train,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315,0,0
1,1,1,2010-02-12,46039.49,True,train,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315,0,1
2,1,1,2010-02-19,41595.55,False,train,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315,0,0
3,1,1,2010-02-26,19403.54,False,train,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315,0,0
4,1,1,2010-03-05,21827.90,False,train,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315,0,0


Hoặc ta có thể sử dụng mapping để mã hóa giá trị tương ứng tùy ý cho mỗi giá trị của biến trong mã hóa nhãn:

In [38]:
mapping = {
    'A' : 1,
    'B' : 2,
    'C' : 3
}
df['Type_enc'] = df['Type'].map(mapping)

In [39]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Set,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Type_enc,IsHoliday_enc
0,1,1,2010-02-05,24924.50,False,train,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315,1,0
1,1,1,2010-02-12,46039.49,True,train,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315,1,1
2,1,1,2010-02-19,41595.55,False,train,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315,1,0
3,1,1,2010-02-26,19403.54,False,train,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315,1,0
4,1,1,2010-03-05,21827.90,False,train,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315,1,0


**Ưu điểm:** Đơn giản, nhanh chóng.

**Nhược điểm** Giả định mối quan hệ thứ tự giữa các nhãn, có thể gây nhầm lẫn cho các mô hình Linear Regression. Không phù hợp với dữ liệu phi thứ tự.

# 2. One-Hot Encoding

One-hot Encoding là chiến thuật mã hóa dữ liệu dạng phân loại bằng cách tách mỗi giá trị của biến thành một cột riêng biệt và gán giá trị cho cột đó là 0 (tương đương dòng dữ liệu không nhận giá trị của cột mới trong biến ban đầu) hoặc 1 (tương đương với dòng dữ liệu nhận giá trị của cột mới trong biến ban đầu).

Trong bộ dữ liệu này, ta có thể mã hóa biến `Type` với các giá trị A, B, C thành các cột `Type_A`, `Type_B`, `Type_C` với các giá trị 0 và 1.
Cũng có thể mã hóa biến `IsHoliday` với các giá trị True và False thành hai cột `Holiday` và `NotHoliday` với các giá trị 0, 1. Tuy nhiên với biến phân loại nhị phân, hành động này có vẻ khiến cho dữ liệu và mô hình trở nên phức tạp hơn và không cần thiết. Ta cũng có thể mã hóa luôn cho biến `Set` đã tạo trước đó thành hai cột `Train_Set` và `Test_Set`, tuy nhiên mặc dù biến này không nhận giá trị numerical nhưng nó không đóng góp vào huấn luyện model nên không cần mã hóa. 

In [40]:
# Mã hóa one-hot cho biến Type:
df = pd.get_dummies(df, columns=['Type'], prefix='Type')


In [41]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Set,Temperature,Fuel_Price,MarkDown1,MarkDown2,...,MarkDown4,MarkDown5,CPI,Unemployment,Size,Type_enc,IsHoliday_enc,Type_A,Type_B,Type_C
0,1,1,2010-02-05,24924.50,False,train,42.31,2.572,NaN,NaN,...,NaN,NaN,211.096358,8.106,151315,1,0,True,False,False
1,1,1,2010-02-12,46039.49,True,train,38.51,2.548,NaN,NaN,...,NaN,NaN,211.242170,8.106,151315,1,1,True,False,False
2,1,1,2010-02-19,41595.55,False,train,39.93,2.514,NaN,NaN,...,NaN,NaN,211.289143,8.106,151315,1,0,True,False,False
3,1,1,2010-02-26,19403.54,False,train,46.63,2.561,NaN,NaN,...,NaN,NaN,211.319643,8.106,151315,1,0,True,False,False
4,1,1,2010-03-05,21827.90,False,train,46.50,2.625,NaN,NaN,...,NaN,NaN,211.350143,8.106,151315,1,0,True,False,False


**Ưu điểm:** Không giả định thứ tự giữa các giá trị.

**Nhược điểm:** Tăng số chiều của bộ dữ liệu, khiến dữ liệu trở nên phức tạp hơn.

# 3. Binary Encoding

Ngoài hai chiến thuật mã hóa trên, ta còn có chiến thuật mã hóa nhị phân (Binary Encoding). Dành cho các biến phân loại có số lượng lớn giá trị khác nhau. Cách hoạt động: Mã hóa từng nhãn thành số nhị phân, sau đó chuyển từng bit thành cột.

Trong bộ dữ liệu này, ta có biến `Store` phù hợp với kiểu mã hóa này, tuy nhiên vì biến này đã nhận giá trị numerical và không tương quan nhiều với biến mục tiêu nên ta không cần thiết phải mã hóa. 

# Nhận xét, so sánh các chiến thuật:

Như vậy, ta có các chiến thuật mã hóa là *Label Encoding*, *One-Hot Encoding* và *Binary Encoding*. Ta cũng đã thực hiện mã hóa trên bộp dữ liệu Walmart Recruiting Store Sales Forecating, tuy chưa thực hiện đánh giá hiệu quả của chúng qua thực nghiệm với các mô hình máy học. Tuy nhiên có thể so sánh đơn giản rằng:

- *Label Encoding* đơn giản, dễ thực hiện, không làm tăng chiều của dữ liệu, tuy nhiên lại giả định thứ tự giữa các giá trị của biến, dễ gây nhầm lẫn về mối quan hệ thứ tự cho model, đặc biệt đối với các mô hình tuyến tính và các dữ liệu phi tuyến.
- *One-Hot Encoding* cũng đơn giản, dễ thực hiện, không giả định thứ tự giữa các giá trị, tuy nhiên lại làm tăng chiều dữ liệu, nếu biến có nhiều giá trị khác nhau thì số chiều dữ liệu tăng lên là đáng kể, khiến cho dữ liệu và mô hình trở nên phức tạp hơn.
- *Binary Encoding* khá phức tạp nhưng có thể phù hợp với các biến phân loại có nhiều giá trị, trong các hoàn cảnh cụ thể.